### Camada Silver: Limpeza e Transformação

Este notebbok tem como objetivo aplicar transformações, desnormalizar e mascarar os dados na camada Silver. Com particionamento para melhorar o desempenho de leitura e escrita.

In [0]:
# Importar as bibliotecas necessárias
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Iniciar a SparkSession com configurações otimizadas
spark = SparkSession.builder \
    .appName("Transformação Data Silver") \
    .config("spark.sql.shuffle.partitions", "200")  \
    .config("spark.sql.files.maxPartitionBytes", "128MB") \
    .config("spark.sql.parquet.compression.codec", "snappy") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

# Define um número fixo de partições para shuffle, melhorando o paralelismo                 
# Define o tamanho máximo de partições para evitar muitos arquivos pequenos        
# Usa o codec Snappy para compressão rápida, otimizando tempo de leitura e escrita    
# Habilita otimizações adaptativas, ajustando o número de partições dinamicamente com base no tamanho dos dados

# Definir caminhos de armazenamento no Data Lake
# Ler dados na Bronze e armazenar Silver

bronze_path = "/mnt/lhdw/bronze/vendas"
silver_path = "/mnt/lhdw/silver/vendas"



###Ler o dados da camada Bronze para transformação na camada Silver

In [0]:
# Ler dados da camada Bronze
df_bronze = spark.read.format("parquet").load(bronze_path)
#display(df_bronze)

### Limpeza dos dados

O processo de limpeza que será executado a seguir, transforma o DataFrame df_bronze em df_silver, aplicando diversas manipulações para limpeza e padronização dos dados. Primeiro, ele converte a coluna de data para o formato padrão YYYY-MM-DD, garantindo que os valores sejam corretamente interpretados. Em seguida, extrai e limpa o e-mail, removendo caracteres indesejados e convertendo para letras minúsculas, enquanto o nome é reorganizado para seguir o padrão Nome Sobrenome. A cidade é ajustada para exibir apenas o nome principal, descartando informações extras. Os valores monetários, como Preço Unitário e Custo Unitário, são formatados para ter duas casas decimais, e o Total de Vendas é calculado multiplicando o preço unitário pela quantidade vendida. Para otimizar o DataFrame, as colunas EmailNome e IdCampanha são removidas, pois seus dados já foram extraídos. 

In [0]:
from pyspark.sql.functions import format_number

# Realizar transformações necessárias, incluindo a manipulação do campo EmailNome IdCampanha
df_silver = df_bronze.withColumn("Data", to_date(col("Data"), "yyyy-MM-dd")) \
                     .withColumn("Email", lower(expr("regexp_replace(split(EmailNome, ':')[0], '[()]', '')"))) \
                     .withColumn("Nome", expr("split(split(EmailNome, ':')[1], ', ')")) \
                     .withColumn("Nome", expr("concat(Nome[1], ' ', Nome[0])")) \
                     .withColumn("Cidade", expr("split(Cidade, ',')[0]")) \
                     .withColumn("PrecoUnitario", format_number(col("PrecoUnitario"), 2)) \
                     .withColumn("CustoUnitario", format_number(col("CustoUnitario"), 2)) \
                     .withColumn("TotalVendas", format_number(col("PrecoUnitario") * col("Unidades"),2))\
                     .drop("EmailNome")\
                     .drop("IdCampanha")   
                     

display(df_silver)




###Mascaramento de Dados

O mascaramento de dados é uma técnica essencial para garantir a privacidade e segurança das informações sensíveis. Existem diferentes abordagens para ocultar ou substituir dados sem comprometer sua estrutura original.

###Técnicas de Mascaramento:

- **Anonimização**: Substitui valores reais por pseudônimos ou identificadores, garantindo que os dados originais não possam ser associados diretamente ao usuário.  
  _Exemplo:_ Trocar `"João Silva"` por `"Cliente123"`.

- **Hashing**: Aplica funções de hash para esconder informações, como e-mails, garantindo que o valor original não seja acessível.  
  _Exemplo:_ `hash('email@domain.com')`.

- **Substituição**: Utiliza valores fictícios gerados por bibliotecas como **Faker**, mantendo o formato dos dados sem expor informações reais.  
  _Exemplo:_ `"joao@gmail.com"` ➝ `"ficticio@email.com"`.

- **Cifragem (Encryption)**: Aplica criptografia em campos sensíveis para proteger os dados armazenados, impedindo que sejam acessados sem a chave correta.

Essas práticas são fundamentais para garantir conformidade com normas de proteção de dados e preservar a privacidade dos usuário


### Implementação com PySpark:

In [0]:
from pyspark.sql.functions import sha2, concat_ws

df_silver = df_silver.withColumn("Nome", sha2(col("Nome"), 256)) \
                     .withColumn("Email", sha2(col("Email"), 256))

display(df_silver)


In [0]:
del df_bronze
#Limpeza do df_bronze já que não será mais necessário

### Gravando as transformações da Camada Silver

O particionamento por ano e mês, será feito para otimizar as consultas baseadas em data, com recomendação de tamanho de arquivo em formato Parquet.

In [0]:
# Particionamento por ano e mês para otimizar consultas baseadas em data, com recomendação de tamanho de arquivo

df_silver.withColumn("Ano", year("Data")) \
         .withColumn("Mes", month("Data")) \
         .write.option("maxRecordsPerFile", 50000) \
         .partitionBy("Ano", "Mes") \
         .format("parquet") \
         .mode("overwrite") \
         .save(silver_path)
#Contagem de registros
df_silver.count()

%md
**Justificativa para particionamento:**

partitionBy("Ano", "Mes"): Particionar os dados pelas coluna Ano e Mês ajuda a otimizar a leitura quando queremos filtrar ou consultar dados baseados em periodos específicos. Isso reduz o número de arquivos escaneados em consultas, melhorando a performance.



### Limpando a Memória

In [0]:
import gc
gc.collect()

In [0]:
del df_silver